# Can ModernBERT's own `yes` / `no` be the JEPA targets?

In `decoder_jepa.ipynb` the target side would become just two vectors: the encoding of `yes` and the encoding of `no` (no state, no question). The online side predicts ẑ from the state and question, and the model scores each answer as

`softmax(cos(ẑ, t_yes), cos(ẑ, t_no)) / τ`

If the two targets are almost the same vector, even a perfect ẑ cannot express a confident answer. With cosine `c` between them, the most probability the gold can get is

**p_max = sigmoid((1 − c) / τ)**

At τ = 0.1, a gold of 0.9 needs `c ≤ 0.78`.

This notebook measures `c` for many ways of building the two targets, without any training:

- **Word pairs:** `yes`/`no`, `true`/`false` and a few others.
- **Repeats:** the word once, or repeated N times (`yes yes yes …`).
- **Separator:** a space or `, ` between the repeats.
- **Pooling:** the mean over all tokens, the mean without `[CLS]` and `[SEP]`, or the `[CLS]` vector alone.
- **Layer:** the last layer, or a middle layer.
- **Space:** ModernBERT's 768-d output, or after the JEPA projection head at its starting weights (768 → 256), as in `decoder_jepa.ipynb`.

The result is a table ranked by cosine, with p_max and a pass/fail against the gate.

## Settings

In [ ]:
import os

def setting(name, default):
    value = os.environ.get(name.upper())
    if value is None:
        return default
    if isinstance(default, list):
        return [v.strip() for v in value.split(",") if v.strip()]   # lists: comma-separated
    return type(default)(value)

CONFIG = {
    "model_name": setting("model_name", "answerdotai/ModernBERT-base"),
    "seed": setting("seed", 42),
    "temperature": setting("temperature", 0.1),
    "gold_to_reach": setting("gold_to_reach", 0.9),       # the gate: p_max must reach this
    "latent_dim": setting("latent_dim", 256),             # the JEPA projection, as in decoder_jepa.ipynb
    # pairs as "positive/negative"; an environment override separates them with ","
    "pairs": setting("pairs", ["yes/no", "true/false", "Yes/No", "YES/NO",
                               "correct/incorrect", "accept/reject", "agree/disagree"]),
    "repeats": [int(n) for n in setting("repeats", ["1", "2", "4", "8", "16", "32"])],
    "separators": setting("separators", ["space", "comma"]),
    "poolings": setting("poolings", ["mean_all", "mean_words", "cls"]),
    "layers": setting("layers", ["last", "middle"]),
    "out_dir": setting("out_dir", "target_check"),
}

SEPARATORS = {"space": " ", "comma": ", "}

for k, v in CONFIG.items():
    print(f"{k:15s}: {v}")

## The model

ModernBERT-base exactly as `decoder_jepa.ipynb` loads it, plus the JEPA projection head (768 → 768 → 256 with GELU) at its starting weights. The projection is random at the start, so it can move the cosine either way; both spaces are reported.

In [ ]:
import math
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
model = AutoModel.from_pretrained(CONFIG["model_name"]).to(device).eval()

hidden = model.config.hidden_size
projection = nn.Sequential(
    nn.Linear(hidden, hidden),
    nn.GELU(),
    nn.Linear(hidden, CONFIG["latent_dim"]),
).to(device).eval()

n_layers = model.config.num_hidden_layers
LAYER_INDEX = {"last": n_layers, "middle": n_layers // 2}   # hidden_states[0] is the embedding layer

print(f"device {device} | layers {n_layers} | middle = layer {LAYER_INDEX['middle']}")

## Encoding one target

The text is the word repeated N times. Pooling:
- **mean_all:** the mean over every token, `[CLS]` and `[SEP]` included (what `decoder_jepa.ipynb`'s encoder does now).
- **mean_words:** the mean over the word tokens only, `[CLS]` and `[SEP]` left out.
- **cls:** the `[CLS]` vector alone.

In [ ]:
def target_text(word, repeats, separator):
    return SEPARATORS[separator].join([word] * repeats)


@torch.no_grad()
def encode(text, pooling, layer):
    batch = tokenizer(text, return_tensors="pt", return_special_tokens_mask=True).to(device)
    special = batch.pop("special_tokens_mask")[0].bool()
    out = model(**batch, output_hidden_states=True)
    states = out.hidden_states[LAYER_INDEX[layer]][0]          # [tokens, hidden]

    if pooling == "mean_all":
        vector = states.mean(0)
    elif pooling == "mean_words":
        vector = states[~special].mean(0)
    elif pooling == "cls":
        vector = states[0]
    else:
        raise ValueError(pooling)

    return vector.float(), int((~special).sum())


def p_max(cosine, temperature=CONFIG["temperature"]):
    return 1.0 / (1.0 + math.exp(-(1.0 - cosine) / temperature))


# the gate: the largest cosine that still lets the gold reach `gold_to_reach`
GATE = 1.0 - CONFIG["temperature"] * math.log(CONFIG["gold_to_reach"] / (1 - CONFIG["gold_to_reach"]))
print(f"gate: cosine <= {GATE:.3f} (gold {CONFIG['gold_to_reach']} at tau {CONFIG['temperature']})")

v, n = encode("yes", "mean_words", "last")
print(f"check: 'yes' -> {n} word token(s), vector {tuple(v.shape)}")

## The reference: how alike are unrelated words?

Mean-pooled BERT vectors are all fairly alike, so a cosine only means something next to a baseline. The same settings are applied to pairs of unrelated words; the yes/no pair should be clearly **below** this.

In [ ]:
UNRELATED = [("invoice", "giraffe"), ("server", "banana"), ("refund", "mountain"), ("driver", "purple")]


def cosine_pair(a, b, repeats, separator, pooling, layer, projected):
    va, _ = encode(target_text(a, repeats, separator), pooling, layer)
    vb, _ = encode(target_text(b, repeats, separator), pooling, layer)
    if projected:
        with torch.no_grad():
            va, vb = projection(va), projection(vb)
    return F.cosine_similarity(va, vb, dim=0).item()

## All combinations

In [ ]:
import itertools

rows = []

combos = list(itertools.product(CONFIG["repeats"], CONFIG["separators"], CONFIG["poolings"],
                                CONFIG["layers"], [False, True]))

for repeats, separator, pooling, layer, projected in combos:

    if repeats == 1 and separator != CONFIG["separators"][0]:
        continue                                   # a single word has no separator

    reference = np.mean([cosine_pair(a, b, repeats, separator, pooling, layer, projected)
                         for a, b in UNRELATED])

    for pair in CONFIG["pairs"]:
        positive, negative = pair.split("/")
        c = cosine_pair(positive, negative, repeats, separator, pooling, layer, projected)
        rows.append({
            "pair": pair,
            "repeats": repeats,
            "separator": separator,
            "pooling": pooling,
            "layer": layer,
            "space": "projected_256" if projected else "backbone_768",
            "cosine": round(c, 4),
            "unrelated_cosine": round(float(reference), 4),
            "margin_vs_unrelated": round(float(reference) - c, 4),
            "p_max": round(p_max(c), 4),
            "passes": c <= GATE,
        })

results = pd.DataFrame(rows).sort_values("cosine").reset_index(drop=True)
print(f"{len(results)} combinations, {int(results['passes'].sum())} pass the gate (cosine <= {GATE:.3f})")

## Best combinations

In [ ]:
pd.set_option("display.width", 200)
display(results.head(25))

## Summaries by each choice

In [ ]:
for column in ["pair", "repeats", "pooling", "layer", "space", "separator"]:
    summary = results.groupby(column).agg(
        best_cosine=("cosine", "min"),
        median_cosine=("cosine", "median"),
        passing=("passes", "sum"),
        combos=("passes", "size"),
    ).sort_values("best_cosine")
    print(f"\nby {column}")
    display(summary)

## The current setting and the plain word

For comparison: what `decoder_jepa.ipynb`'s encoder would give for a single `yes` / `no` today (mean over all tokens, last layer, projected), and the same word pair with the repeat and pooling changes.

In [ ]:
focus = results[(results["pair"].isin(["yes/no", "true/false"]))
                & (results["layer"] == "last")
                & (results["separator"] == "space")]
display(focus.pivot_table(index=["pair", "pooling", "space"], columns="repeats", values="cosine"))

## Verdict and saved table

In [ ]:
os.makedirs(CONFIG["out_dir"], exist_ok=True)
path = os.path.join(CONFIG["out_dir"], "target_cosines.csv")
results.to_csv(path, index=False)

passing = results[results["passes"]]

if len(passing):
    best = passing.iloc[0]
    print("PASS: ModernBERT's own targets can work with this setting:")
    print(best.to_string())
else:
    best = results.iloc[0]
    print(f"FAIL: no combination reaches cosine <= {GATE:.3f}.")
    print("Use our own fixed target vectors instead (e.g. two opposite unit vectors, cosine -1).")
    print("\nClosest combination:")
    print(best.to_string())

print(f"\nSaved: {path}")